In [54]:
import pandas as pd
books = pd.read_csv('C:/Users/Admin/Downloads/archive/books.csv')
tags = pd.read_csv('C:/Users/Admin/Downloads/archive/tags.csv')
book_tags = pd.read_csv('C:/Users/Admin/Downloads/archive/book_tags.csv')

In [3]:
books.head(3)

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...


In [37]:
sl_sach = books['book_id'].nunique()
sl_sach

10000

In [4]:
tags.head(3)

,tag_id,tag_name
0,0,-
1,1,--1-
2,2,--10-


In [6]:
book_tags.head(13)

,goodreads_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173
3,1,8717,12986
4,1,33114,12716
5,1,11743,9954
6,1,14017,7169
7,1,5207,6221
8,1,22743,4974
9,1,32989,4364


In [11]:
book_tags.describe()

,book_id,tag_id,count
count,9.999120e+05,999912.000000,999912.000000
mean,5.263442e+06,16324.527073,208.869633
std,7.574057e+06,9647.846196,3501.265173
min,1.000000e+00,0.000000,-1.000000
25%,4.622700e+04,8067.000000,7.000000
50%,3.948410e+05,15808.000000,15.000000
75%,9.378297e+06,24997.000000,40.000000
max,3.328864e+07,34251.000000,596234.000000


In [ ]:
book_tags = book_tags.rename(columns={'goodreads_book_id': 'book_id'})

In [24]:



# ---------------------------------------------------------
# YÊU CẦU 2: Lọc ra các tag_id có số đếm "đáng kể"
# ---------------------------------------------------------

# Bước a: Tính tổng số 'count' của TỪNG quyển sách (theo book_id)
# transform('sum') sẽ tạo ra một cột tạm thời chứa tổng count của cuốn sách tương ứng với từng dòng
tong_count_tung_sach = book_tags.groupby('book_id')['count'].transform('sum')

# Bước b: Đặt ra một ngưỡng (threshold) để đánh giá mức độ "đáng kể"
# Ví dụ ở đây: Số count của tag_id đó phải chiếm ít nhất 5% (0.05) tổng số count của quyển sách
nguong_phan_tram = 0.02

# Bước c: Lọc DataFrame
df_filtered = book_tags[book_tags['count'] >= (tong_count_tung_sach * nguong_phan_tram)]

# (Tùy chọn) Reset lại index sau khi lọc cho gọn gàng
df_filtered = df_filtered.reset_index(drop=True)

# Xem kết quả
df_filtered.describe()

,book_id,tag_id,count
count,5.341800e+04,53418.000000,53418.000000
mean,5.024860e+06,18434.226422,3209.922872
std,7.356218e+06,9589.651018,14821.335043
min,1.000000e+00,71.000000,7.000000
25%,4.418600e+04,9124.000000,174.000000
50%,3.491915e+05,15359.000000,402.000000
75%,8.533018e+06,30358.000000,1217.750000
max,3.328864e+07,34249.000000,596234.000000


In [25]:
df_filtered.to_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/test.ipynbbook_tags_filtered.csv', index=False)

In [26]:
# Đếm số lượng giá trị khác nhau trong cột 'book_id'
so_luong_sach = book_tags['book_id'].nunique()
so_luong_sach_1 = df_filtered['book_id'].nunique()
print(f"Có tổng cộng {so_luong_sach} đầu sách khác nhau trong dữ liệu.")
print(f"Có tổng cộng {so_luong_sach_1} đầu sách khác nhau sau khi lọc.")

Có tổng cộng 10000 đầu sách khác nhau trong dữ liệu.
Có tổng cộng 10000 đầu sách khác nhau sau khi lọc.


In [27]:
book_tags_filtered=pd.read_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/test.ipynbbook_tags_filtered.csv')
book_tags_filtered.head(3)

,book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173


In [28]:
# Thực hiện ghép (merge) bảng book_tags_filtered với bảng tags
# on='tag_id': chỉ định cột chung để so khớp
# how='left': giữ lại toàn bộ dữ liệu ở bảng bên trái (book_tags_filtered) 
#             và chỉ lấy thêm tên từ bảng bên phải (tags)
book_tags_new = pd.merge(book_tags_filtered, tags, on='tag_id', how='left')

# Kiểm tra kết quả
book_tags_new.head()

,book_id,tag_id,count,tag_name
0,1,30574,167697,to-read
1,1,11305,37174,fantasy
2,1,11557,34173,favorites
3,1,8717,12986,currently-reading
4,1,33114,12716,young-adult


In [29]:
book_tags_new.to_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_tags.csv', index=False)

In [32]:
# Đếm xem mỗi tag_name xuất hiện trên bao nhiêu cuốn sách khác nhau
tag_phu_song_rong = book_tags_new.groupby('tag_name')['book_id'].count().reset_index()

# Đổi tên cột cho dễ hiểu và sắp xếp
tag_phu_song_rong = tag_phu_song_rong.rename(columns={'book_id': 'so_luong_sach'})
tag_phu_song_rong = tag_phu_song_rong.sort_values(by='so_luong_sach', ascending=False).reset_index(drop=True)

print("--- TOP 10 TAGS XUẤT HIỆN TRÊN NHIỀU SÁCH NHẤT ---")
print(tag_phu_song_rong.head(20))

--- TOP 10 TAGS XUẤT HIỆN TRÊN NHIỀU SÁCH NHẤT ---
              tag_name  so_luong_sach
0              to-read           9803
1    currently-reading           5810
2              fiction           3907
3            favorites           3301
4              fantasy           1898
5              romance           1269
6          young-adult           1249
7              mystery           1102
8          non-fiction            904
9                owned            848
10         books-i-own            799
11              series            776
12            classics            757
13  historical-fiction            744
14                  ya            667
15            thriller            575
16          paranormal            534
17     science-fiction            531
18           childrens            495
19              sci-fi            482


In [34]:
books_tags=pd.read_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_tags.csv')

# Thực hiện gom nhóm theo book_id và gộp tag_name thành list
book_tags_list = books_tags.groupby('book_id')['tag_name'].apply(list).reset_index()

# Đổi tên cột cho rõ nghĩa (tùy chọn)
book_tags_list = book_tags_list.rename(columns={'tag_name': 'tag_list'})

# Kiểm tra kết quả
book_tags_list.head(10)

,book_id,tag_list
0,1,"[to-read, fantasy, favorites, currently-readin..."
1,2,"[to-read, currently-reading, fantasy, favorite..."
2,3,"[to-read, favorites, fantasy, currently-reading]"
3,5,"[favorites, fantasy, currently-reading, young-..."
4,6,"[fantasy, young-adult, fiction, harry-potter, ..."
5,8,"[to-read, favorites, fantasy]"
6,10,"[to-read, favorites, fantasy, currently-reading]"
7,11,"[to-read, currently-reading, science-fiction, ..."
8,13,"[to-read, currently-reading, favorites, scienc..."
9,21,"[to-read, currently-reading, history, nonficti..."


In [38]:
books.columns

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url'],
      dtype='object')

In [56]:
# Điền các giá trị thiếu của original_title bằng giá trị của title ở cùng dòng
books['original_title'] = books['original_title'].fillna(books['title'])

In [58]:
columns = ['id','best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13','title', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count']
books=books.drop(columns, axis=1)
books.head(10)

,book_id,authors,original_publication_year,original_title,language_code,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...
5,11870085,John Green,2012.0,The Fault in Our Stars,eng,47994,92723,327550,698471,1311871,https://images.gr-assets.com/books/1360206420m...,https://images.gr-assets.com/books/1360206420s...
6,5907,J.R.R. Tolkien,1937.0,The Hobbit or There and Back Again,en-US,46023,76784,288649,665635,1119718,https://images.gr-assets.com/books/1372847500m...,https://images.gr-assets.com/books/1372847500s...
7,5107,J.D. Salinger,1951.0,The Catcher in the Rye,eng,109383,185520,455042,661516,709176,https://images.gr-assets.com/books/1398034300m...,https://images.gr-assets.com/books/1398034300s...
8,960,Dan Brown,2000.0,Angels & Demons,en-CA,77841,145740,458429,716569,680175,https://images.gr-assets.com/books/1303390735m...,https://images.gr-assets.com/books/1303390735s...
9,1885,Jane Austen,1813.0,Pride and Prejudice,eng,54700,86485,284852,609755,1155673,https://images.gr-assets.com/books/1320399351m...,https://images.gr-assets.com/books/1320399351s...


In [59]:
# Gộp tag_list từ book_tags_list vào bảng books
# on='book_id': Khớp lệnh dựa trên mã sách
# how='left': Giữ lại tất cả sách trong bảng books, 
#             nếu sách nào không có tag thì giá trị sẽ là NaN
bookss = pd.merge(books, book_tags_list, on='book_id', how='left')
bookss.head()

,book_id,authors,original_publication_year,original_title,language_code,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,tag_list
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,"[favorites, currently-reading, young-adult, fi..."
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,"[to-read, favorites, fantasy, currently-reading]"
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,"[young-adult, fantasy, favorites, vampires, ya..."
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,"[classics, favorites, to-read, classic, histor..."
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,"[classics, favorites, fiction, classic, books-..."


In [60]:


# Xử lý các dòng không có tag (NaN) thành một danh sách trống []
# Điều này giúp code không bị lỗi khi bạn thực hiện các thao tác xử lý chuỗi/list sau này
bookss['tag_list'] = bookss['tag_list'].apply(lambda d: d if isinstance(d, list) else [])

# Kiểm tra 5 dòng đầu tiên
bookss.head()

,book_id,authors,original_publication_year,original_title,language_code,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,tag_list
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,"[favorites, currently-reading, young-adult, fi..."
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,"[to-read, favorites, fantasy, currently-reading]"
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,"[young-adult, fantasy, favorites, vampires, ya..."
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,"[classics, favorites, to-read, classic, histor..."
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,"[classics, favorites, fiction, classic, books-..."


In [65]:
bookss = bookss.rename(columns={'tag_list': 'tags'})
bookss.head()

,book_id,authors,original_publication_year,original_title,language_code,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,tags
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,"[favorites, currently-reading, young-adult, fi..."
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...,"[to-read, favorites, fantasy, currently-reading]"
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...,"[young-adult, fantasy, favorites, vampires, ya..."
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...,"[classics, favorites, to-read, classic, histor..."
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...,"[classics, favorites, fiction, classic, books-..."


In [67]:
bookss.columns  


Index(['book_id', 'authors', 'original_publication_year', 'original_title',
       'language_code', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4',
       'ratings_5', 'image_url', 'small_image_url', 'tags'],
      dtype='object')

In [66]:
# Đếm số lượng giá trị null trong mỗi cột
so_luong_null = bookss.isna().sum()

print(so_luong_null)

book_id                         0
authors                         0
original_publication_year      21
original_title                  0
language_code                1084
ratings_1                       0
ratings_2                       0
ratings_3                       0
ratings_4                       0
ratings_5                       0
image_url                       0
small_image_url                 0
tags                            0
dtype: int64


In [68]:
bookss= bookss[['book_id', 'authors', 'original_publication_year', 'original_title',
       'language_code', 'tags', 'ratings_1', 'ratings_2', 'ratings_3', 'ratings_4',
       'ratings_5', 'image_url', 'small_image_url']]

In [79]:
bookss.describe()

,book_id,original_publication_year,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5
count,1.000000e+04,9979.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,1.000000e+04
mean,5.264697e+06,1981.987674,1345.040600,3110.885000,11475.893800,1.996570e+04,2.378981e+04
std,7.575462e+06,152.576665,6635.626263,9717.123578,28546.449183,5.144736e+04,7.976889e+04
min,1.000000e+00,-1750.000000,11.000000,30.000000,323.000000,7.500000e+02,7.540000e+02
25%,4.627575e+04,1990.000000,196.000000,656.000000,3112.000000,5.405750e+03,5.334000e+03
50%,3.949655e+05,2004.000000,391.000000,1163.000000,4894.000000,8.269500e+03,8.836000e+03
75%,9.382225e+06,2011.000000,885.000000,2353.250000,9287.000000,1.602350e+04,1.730450e+04
max,3.328864e+07,2017.000000,456191.000000,436802.000000,793319.000000,1.481305e+06,3.011543e+06


In [69]:
bookss.to_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_with_tags.csv', index=False)

In [7]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import uuid

# ==========================================
# PHẦN 1: CẤU HÌNH THÔNG TIN (BẠN CẦN THAY ĐỔI)
# ==========================================

# 1. Đường dẫn đến file CSV
DUONG_DAN_FILE = r'E:\CodeFolder\Github_project\BookRecProject\Backend\books_with_tags.csv'

# 2. Tên Server SQL của bạn 
TEN_SERVER = r'LEHIEU\SQLEXPRESS' 
TEN_DATABASE = 'BookRecDb'

# 4. Tên bảng trong Database
TEN_BANG = 'Books' 
TEN_SCHEMA = 'dbo'

# 5. Cấu hình Driver SQL Server
DRIVER = 'ODBC Driver 17 for SQL Server' 


# ==========================================
# PHẦN 2: CHUỖI KẾT NỐI (CHỌN 1 TRONG 2 CÁCH)
# ==========================================

# CÁCH 1: Dùng Windows Authentication (Đăng nhập không cần pass) - KHUYÊN DÙNG
params = urllib.parse.quote_plus(
    f"DRIVER={DRIVER};"
    f"SERVER={TEN_SERVER};"
    f"DATABASE={TEN_DATABASE};"
    f"Trusted_Connection=yes;"
)


# ==========================================
# PHẦN 3: THỰC THI (KHÔNG CẦN SỬA)
# ==========================================

def run_import():
    print(f"1. Đang đọc file CSV từ: {DUONG_DAN_FILE}...")
    try:
        # on_bad_lines='skip': Bỏ qua các dòng bị lỗi lệch cột do dấu phẩy
        df = pd.read_csv(DUONG_DAN_FILE, on_bad_lines='skip')
    except Exception as e:
        print(f"❌ LỖI ĐỌC FILE CSV: {e}")
        return

    # Kiểm tra xem file có dữ liệu không
    if df.empty:
        print("❌ LỖI: DataFrame rỗng! Hãy kiểm tra lại đường dẫn file hoặc cấu trúc file CSV.")
        return
        
    so_dong = len(df)
    print(f"✅ Đã đọc thành công {so_dong} dòng dữ liệu.")
    
    # ---------------------------------------------------------
    # XỬ LÝ DỮ LIỆU: CHUYỂN UUID VÀ XÓA LỖI NULL
    # ---------------------------------------------------------
    print("-> Đang xử lý dữ liệu (Chuyển UUID và điền dữ liệu trống)...")
    try:
        # 1. Chuyển đổi book_id sang chuẩn UUID của SQL Server
        df['book_id'] = df['book_id'].apply(lambda x: str(uuid.UUID(int=int(x))))
        
        # 2. Xử lý các cột bị trống (NULL) để tránh lỗi Database từ chối
        df['language_code'] = df['language_code'].fillna('unknown')
        df['original_title'] = df['original_title'].fillna('Unknown Title')
        df['authors'] = df['authors'].fillna('Unknown Author')
        df['tags'] = df['tags'].fillna('[]')
        df['image_url'] = df['image_url'].fillna('')
        df['small_image_url'] = df['small_image_url'].fillna('')
        df['original_publication_year'] = df['original_publication_year'].fillna(0)

    except Exception as e:
        print(f"❌ LỖI XỬ LÝ DỮ LIỆU: {e}")
        return
    # ---------------------------------------------------------

    print("\n--- Mẫu dữ liệu (Dòng 1) sau khi xử lý ---")
    print(df.iloc[0])
    print("------------------------------------------\n")

    print("2. Đang thiết lập kết nối đến SQL Server...")
    try:
        engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
    except Exception as e:
        print(f"❌ LỖI KẾT NỐI: {e}")
        return

    print("3. Đang đẩy dữ liệu vào Database. Vui lòng chờ...")
    try:
        # Đẩy dữ liệu. if_exists='append' nghĩa là chèn thêm vào bảng đã có.
        df.to_sql(TEN_BANG, con=engine, if_exists='append', index=False, schema=TEN_SCHEMA)
        
        # Xác minh lại dữ liệu đã thực sự vào DB chưa
        with engine.connect() as conn:
            result = conn.execute(f"SELECT COUNT(*) FROM {TEN_SCHEMA}.{TEN_BANG}")
            count_db = result.scalar()
            print(f"✅ QUÁ TRÌNH IMPORT HOÀN TẤT THÀNH CÔNG!")
            print(f"📊 Số dòng hiện tại trong bảng {TEN_SCHEMA}.{TEN_BANG} là: {count_db}")
            
    except Exception as e:
        print(f"❌ LỖI KHI PUSH VÀO DATABASE: {e}")

# Chạy hàm
if __name__ == "__main__":
    run_import()

1. Đang đọc file CSV từ: E:\CodeFolder\Github_project\BookRecProject\Backend\books_with_tags.csv...
✅ Đã đọc thành công 10000 dòng dữ liệu.
-> Đang xử lý dữ liệu (Chuyển UUID và điền dữ liệu trống)...

--- Mẫu dữ liệu (Dòng 1) sau khi xử lý ---
book_id                                   00000000-0000-0000-0000-0000002a38cc
authors                                                        Suzanne Collins
original_publication_year                                               2008.0
original_title                                                The Hunger Games
language_code                                                              eng
tags                         ['favorites', 'currently-reading', 'young-adul...
ratings_1                                                                66715
ratings_2                                                               127936
ratings_3                                                               560092
ratings_4                                   

In [1]:
import pandas as pd
import numpy as np
books_with_tags = pd.read_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_with_tags_enhanced.csv')


In [2]:

books_with_tags.head()

,book_id,authors,original_publication_year,original_title,language_code,tags,ratings_1,ratings_2,ratings_3,ratings_4,...,price,mood,badge,description,longDescription,pages,readTime,status,chapters,previewText
0,2767052,Suzanne Collins,2008.0,The Hunger Games,eng,"['favorites', 'currently-reading', 'young-adul...",66715,127936,560092,1481305,...,44000,['adventurous'],Trending,Tóm tắt ngắn gọn về nội dung và thông điệp chí...,"Mô tả chi tiết về bối cảnh, tuyến nhân vật và ...",0,0,complete,0,"Trích đoạn chương mở đầu... ""Vào một ngày nọ, ..."
1,3,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,eng,"['to-read', 'favorites', 'fantasy', 'currently...",75504,101676,455024,1156318,...,34000,['adventurous'],New,Tóm tắt ngắn gọn về nội dung và thông điệp chí...,"Mô tả chi tiết về bối cảnh, tuyến nhân vật và ...",0,0,complete,0,"Trích đoạn chương mở đầu... ""Vào một ngày nọ, ..."
2,41865,Stephenie Meyer,2005.0,Twilight,en-US,"['young-adult', 'fantasy', 'favorites', 'vampi...",456191,436802,793319,875073,...,45000,['adventurous'],New,Tóm tắt ngắn gọn về nội dung và thông điệp chí...,"Mô tả chi tiết về bối cảnh, tuyến nhân vật và ...",0,0,ongoing,0,"Trích đoạn chương mở đầu... ""Vào một ngày nọ, ..."
3,2657,Harper Lee,1960.0,To Kill a Mockingbird,eng,"['classics', 'favorites', 'to-read', 'classic'...",60427,117415,446835,1001952,...,36000,['focused'],New,Tóm tắt ngắn gọn về nội dung và thông điệp chí...,"Mô tả chi tiết về bối cảnh, tuyến nhân vật và ...",0,0,complete,0,"Trích đoạn chương mở đầu... ""Vào một ngày nọ, ..."
4,4671,F. Scott Fitzgerald,1925.0,The Great Gatsby,eng,"['classics', 'favorites', 'fiction', 'classic'...",86236,197621,606158,936012,...,20000,['focused'],New,Tóm tắt ngắn gọn về nội dung và thông điệp chí...,"Mô tả chi tiết về bối cảnh, tuyến nhân vật và ...",0,0,complete,0,"Trích đoạn chương mở đầu... ""Vào một ngày nọ, ..."


In [ ]:
import pandas as pd
import ast

# 1. Nếu dữ liệu trong cột 'tags' đang ở dạng chuỗi (ví dụ: "[tag1, tag2]"), 
# bạn cần chuyển nó về dạng list thực thụ trước:
# books_with_tags['tags'] = books_with_tags['tags'].apply(ast.literal_eval)

# 2. Sử dụng explode để tách mỗi phần tử trong list thành một dòng riêng biệt,
# sau đó dùng unique() để lấy danh sách các tag không trùng lặp.
unique_tags = books_with_tags['tags'].explode().unique()

# 3. Chuyển kết quả về dạng list để dễ quan sát
list_unique_tags = unique_tags.tolist()

print("Các loại tag có trong dữ liệu là:")
print(list_unique_tags)

Các loại tag có trong dữ liệu là:
["['favorites', 'currently-reading', 'young-adult', 'fiction', 'dystopian', 'to-read', 'dystopia', 'fantasy', 'ya', 'science-fiction', 'books-i-own', 'sci-fi']", "['to-read', 'favorites', 'fantasy', 'currently-reading']", "['young-adult', 'fantasy', 'favorites', 'vampires', 'ya', 'fiction', 'to-read', 'paranormal', 'books-i-own', 'vampire', 'twilight']", "['classics', 'favorites', 'to-read', 'classic', 'historical-fiction', 'owned', 'school']", "['classics', 'favorites', 'fiction', 'classic', 'books-i-own', 'owned', 'literature']", "['favorites', 'to-read', 'young-adult', 'fiction', 'ya', 'romance', 'books-i-own', 'contemporary', 'owned']", "['fantasy', 'favorites', 'classics', 'to-read', 'fiction', 'books-i-own']", "['classics', 'favorites', 'fiction', 'to-read', 'classic', 'young-adult', 'books-i-own', 'owned']", "['to-read', 'fiction', 'mystery', 'favorites', 'thriller', 'dan-brown', 'owned', 'books-i-own']", "['classics', 'favorites', 'fiction', 'r

In [11]:
import pandas as pd
import ast
from collections import Counter

# 1. Định nghĩa bộ từ điển ánh xạ Tag -> Mood
mood_mapping = {
    'adventurous': ['fantasy', 'science-fiction', 'sci-fi', 'dystopian', 'dystopia', 'vampires', 'paranormal', 'thriller', 'adventure', 'action', 'magic', 'dragons', 'horror', 'zombies', 'supernatural', 'epic-fantasy', 'survival', 'suspense'],
    'curious': ['mystery', 'crime', 'non-fiction', 'nonfiction', 'history', 'biography', 'memoir', 'memoirs', 'science', 'true-crime', 'psychology', 'philosophy', 'travel', 'detective'],
    'romantic': ['romance', 'chick-lit', 'contemporary-romance', 'paranormal-romance', 'historical-romance', 'erotica', 'new-adult', 'love-triangle', 'lgbt', 'lgbtq'],
    'focused': ['classics', 'historical-fiction', 'literature', 'poetry', 'business', 'economics', 'politics', 'religion', 'spirituality', 'school', 'classic', 'historical'],
    'relaxed': ['humor', 'comedy', 'childrens', 'picture-books', 'comics', 'graphic-novels', 'contemporary', 'animals', 'food', 'fairy-tales', 'middle-grade', 'manga', 'graphic-novel', 'kids']
}

# Đảo ngược từ điển để lookup nhanh hơn: {'fantasy': 'adventurous', 'mystery': 'curious', ...}
tag_to_mood = {}
for mood, tags in mood_mapping.items():
    for tag in tags:
        tag_to_mood[tag] = mood

# 2. Hàm phân loại mood cho từng dòng
def assign_moods(tags_string):
    try:
        # Chuyển chuỗi "['tag1', 'tag2']" thành list Python thật. 
        # Nếu data của bạn đã là list rồi thì bỏ qua dòng ast.literal_eval này, chỉ cần gán tag_list = tags_string
        tag_list = ast.literal_eval(tags_string) 
    except (ValueError, SyntaxError):
        return []

    mood_counter = Counter()
    
    # Duyệt qua từng tag của sách
    for tag in tag_list:
        tag_lower = tag.lower().strip()
        # Nếu tag có trong từ điển, cộng 1 điểm cho mood đó
        if tag_lower in tag_to_mood:
            mood = tag_to_mood[tag_lower]
            mood_counter[mood] += 1
            
    # Lấy ra tối đa 2 mood có điểm cao nhất
    # .most_common(2) trả về dạng [('adventurous', 3), ('romantic', 1)]
    top_moods = [mood for mood, count in mood_counter.most_common(2)]
    
    return top_moods

# 3. Áp dụng hàm vào DataFrame của bạn
# Giả sử df của bạn tên là: books_with_tags

# Tạo cột mới 'mood'
books_with_tags['mood'] = books_with_tags['tags'].apply(assign_moods)

# In thử 10 dòng đầu tiên để kiểm tra kết quả (chỉ in cột tags và mood để dễ nhìn)
print(books_with_tags[['tags', 'mood']].head(10))

                                                tags                    mood
0  ['favorites', 'currently-reading', 'young-adul...           [adventurous]
1  ['to-read', 'favorites', 'fantasy', 'currently...           [adventurous]
2  ['young-adult', 'fantasy', 'favorites', 'vampi...           [adventurous]
3  ['classics', 'favorites', 'to-read', 'classic'...               [focused]
4  ['classics', 'favorites', 'fiction', 'classic'...               [focused]
5  ['favorites', 'to-read', 'young-adult', 'ficti...     [romantic, relaxed]
6  ['fantasy', 'favorites', 'classics', 'to-read'...  [adventurous, focused]
7  ['classics', 'favorites', 'fiction', 'to-read'...               [focused]
8  ['to-read', 'fiction', 'mystery', 'favorites',...  [curious, adventurous]
9  ['classics', 'favorites', 'fiction', 'romance'...     [focused, romantic]


In [18]:
books_with_tags.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   book_id                    10000 non-null  int64  
 1   authors                    10000 non-null  object 
 2   original_publication_year  9979 non-null   float64
 3   original_title             10000 non-null  object 
 4   language_code              8916 non-null   object 
 5   tags                       10000 non-null  object 
 6   ratings_1                  10000 non-null  int64  
 7   ratings_2                  10000 non-null  int64  
 8   ratings_3                  10000 non-null  int64  
 9   ratings_4                  10000 non-null  int64  
 10  ratings_5                  10000 non-null  int64  
 11  image_url                  10000 non-null  object 
 12  small_image_url            10000 non-null  object 
 13  price                      10000 non-null  int3

In [3]:
import pandas as pd
import numpy as np

# Tổng hợp tất cả 20 mã màu (hiện có + bổ sung)
all_accent_colors = [
    # Nhóm Xanh/Tím (Hiện có)
    '#4F46E5', '#7C3AED', '#8B5CF6', '#0EA5E9', '#0284C7',
    # Nhóm Đỏ/Hồng/Cam (Hiện có)
    '#EC4899', '#D97706', '#EA580C',
    # Nhóm Xanh lá/Vàng (Hiện có)
    '#10B981', '#059669', '#CA8A04',
    # Nhóm Tối (Mới bổ sung: Horror, Thriller)
    '#991B1B', '#374151', '#1E3A8A',
    # Nhóm Trung tính (Mới bổ sung: Classic, Non-fiction)
    '#78716C', '#9CA3AF', '#A8A29E',
    # Nhóm Pastel (Mới bổ sung: Children, Light Humor)
    '#FBBF24', '#34D399', '#F472B6'
]

# Tạo cột accentColor và điền ngẫu nhiên mã màu cho từng dòng sách
books_with_tags['accentColor'] = np.random.choice(all_accent_colors, size=len(books_with_tags))

# Kiểm tra thử kết quả (In ra 5 dòng đầu tiên chứa cột id, original_title và accentColor)
print("Kết quả gán màu ngẫu nhiên:")
print(books_with_tags[['book_id', 'original_title', 'accentColor']].head(10))

Kết quả gán màu ngẫu nhiên:
    book_id                            original_title accentColor
0   2767052                          The Hunger Games     #374151
1         3  Harry Potter and the Philosopher's Stone     #1E3A8A
2     41865                                  Twilight     #10B981
3      2657                     To Kill a Mockingbird     #EA580C
4      4671                          The Great Gatsby     #0EA5E9
5  11870085                    The Fault in Our Stars     #10B981
6      5907        The Hobbit or There and Back Again     #0284C7
7      5107                    The Catcher in the Rye     #1E3A8A
8       960                          Angels & Demons      #7C3AED
9      1885                       Pride and Prejudice     #EC4899


In [4]:
books_with_tags.columns

Index(['book_id', 'authors', 'original_publication_year', 'original_title',
       'language_code', 'tags', 'ratings_1', 'ratings_2', 'ratings_3',
       'ratings_4', 'ratings_5', 'image_url', 'small_image_url', 'price',
       'mood', 'badge', 'description', 'longDescription', 'pages', 'readTime',
       'status', 'chapters', 'previewText', 'accentColor'],
      dtype='object')

In [5]:
books_with_tags["accentColor"].isna().sum()

np.int64(0)

## NEXT DAY

In [1]:
import pandas as pd
import numpy as np
books_with_tags = pd.read_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_with_tags_enhanced.csv')

In [2]:
import pandas as pd
import numpy as np

# Giả sử dataframe của bạn tên là: books_with_tags
def generate_smart_prices(df):
    current_year = 2026
    
    # 1. TÍNH TOÁN CÁC CHỈ SỐ CƠ BẢN
    # Tổng số rating (Cộng 5 cột lại, thay 0 bằng 1 để tránh lỗi chia cho 0)
    total_ratings = df[['ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5']].sum(axis=1)
    total_ratings = total_ratings.replace(0, 1) 
    
    # Điểm trung bình (Average Rating)
    avg_rating = (df['ratings_1']*1 + df['ratings_2']*2 + df['ratings_3']*3 + df['ratings_4']*4 + df['ratings_5']*5) / total_ratings
    
    # Năm xuất bản (Fill NaNs bằng năm 2010 để có mức trung bình)
    year = df['original_publication_year'].fillna(2010)
    
    # 2. CHUẨN HÓA CÁC THANG ĐIỂM (Min-Max Scaling từ 0 đến 1)
    # Sách điểm càng cao -> Càng đắt
    rating_score = (avg_rating - avg_rating.min()) / (avg_rating.max() - avg_rating.min() + 1e-9)
    
    # Độ phổ biến (Dùng logarit vì lượt vote thường lệch rất mạnh. Nhiều vote -> Giá nhỉnh hơn)
    pop_log = np.log1p(total_ratings)
    pop_score = (pop_log - pop_log.min()) / (pop_log.max() - pop_log.min() + 1e-9)
    
    # Tuổi thọ sách (Sách càng mới -> Giá càng cao)
    age = current_year - year
    age = np.clip(age, 0, 50) # Coi sách > 50 năm tuổi là giống nhau (sách cổ)
    age_score = 1.0 - (age / 50.0) 
    
    # 3. TẠO ĐIỂM TỔNG HỢP (Raw Score)
    # Trọng số: 40% Điểm số, 30% Độ hot, 30% Độ mới
    # Thêm 1 chút Nhiễu ngẫu nhiên (Noise) để các cuốn sách có cùng chỉ số không bị y hệt giá nhau
    noise = np.random.normal(0, 0.1, size=len(df))
    raw_score = (0.4 * rating_score) + (0.3 * pop_score) + (0.3 * age_score) + noise
    
    # 4. ÉP VÀO PHÂN PHỐI CHUẨN (Z-Score Normalization)
    z_score = (raw_score - raw_score.mean()) / raw_score.std()
    
    # 5. ÁP DỤNG CÔNG THỨC GIÁ TOÁN HỌC
    # Trung bình = 115k. Độ lệch chuẩn = 35k
    # => 68% sách (z_score từ -1 đến 1) sẽ nằm ở 80k - 150k
    prices = 115000 + (z_score * 35000)
    
    # 6. CẮT XÉN (CLIP) VÀ LÀM TRÒN
    # Đảm bảo không có cuốn nào vượt quá giới hạn 45k và 180k
    prices = np.clip(prices, 45000, 180000)
    
    # Làm tròn đến hàng nghìn cho thực tế (ví dụ: 114,321đ -> 114.000đ)
    prices = np.round(prices / 1000) * 1000
    
    return prices.astype(int)

# Gán cột giá mới vào DataFrame
books_with_tags['price'] = generate_smart_prices(books_with_tags)

# ─── KIỂM TRA THÀNH QUẢ ───
# Xem thử phân bố giá xem có đúng 2/3 rơi vào 80k - 150k không nhé:
in_range = books_with_tags['price'].between(80000, 150000).mean()
print(f"Tỉ lệ sách có giá từ 80k - 150k: {in_range * 100:.2f}%")

# Xem thử 5 cuốn đầu tiên
print(books_with_tags[['original_publication_year', 'price']].head(10))

Tỉ lệ sách có giá từ 80k - 150k: 68.75%
   original_publication_year   price
0                     2008.0  180000
1                     1997.0  180000
2                     2005.0  136000
3                     1960.0  115000
4                     1925.0  115000
5                     2012.0  180000
6                     1937.0  142000
7                     1951.0  128000
8                     2000.0  166000
9                     1813.0  168000


In [20]:
import pandas as pd
import numpy as np
from datetime import datetime

# df = books_with_tags.copy()

def simulate_realistic_tracking(row):
    current_year = 2026
    
    # 1. LẤY DỮ LIỆU CƠ BẢN
    r1, r2, r3, r4, r5 = row.get('ratings_1', 0), row.get('ratings_2', 0), row.get('ratings_3', 0), row.get('ratings_4', 0), row.get('ratings_5', 0)
    pub_year = row.get('original_publication_year', current_year)
    price = row.get('price', 115000)
    badge = row.get('badge', '')

    # 2. XỬ LÝ MỎ NEO BẰNG LOGARITHM
    total_ratings = r1 + r2 + r3 + r4 + r5
    if total_ratings < 1: total_ratings = 1
    
    pop_level = np.log10(total_ratings) 
    base_views_7d = (pop_level / 6.0) * np.random.uniform(8000, 20000)
    
    # 3. QUY TẮC TUỔI THỌ & HUY HIỆU (Phạt thật nặng sách cũ để tạo "Đuôi dài chết")
    age = current_year - pub_year
    if age <= 1:
        base_views_7d *= np.random.uniform(1.0, 2.0)  
    elif age <= 3:
        base_views_7d *= np.random.uniform(0.5, 1.0)  
    elif age <= 10:
        base_views_7d *= np.random.uniform(0.005, 0.1) # Rớt thê thảm chỉ còn 0.5% - 10%
    else:
        # Sách quá cũ: Đa số hệ số gần như 0
        base_views_7d *= np.random.uniform(0.0, 0.02) 
        
    if badge in ['Trending', 'Hot']:
        base_views_7d *= 1.5
        
    # SỬ DỤNG POISSON ĐỂ TẠO CÁC SỐ 0 TỰ NHIÊN CHO VIEW
    # VD: Nếu base_views tính ra là 0.7 -> Thuật toán Poisson sẽ cho ra 50% tỉ lệ là 0 view, 35% là 1 view...
    views_7d = np.random.poisson(max(base_views_7d, 0))
    
    # 30 ngày = 7 ngày + random phần còn lại của tháng
    views_30d = views_7d + np.random.poisson(max(base_views_7d * np.random.uniform(2.5, 3.5), 0))

    # 4. TỈ LỆ CHUYỂN ĐỔI BẰNG PHÂN PHỐI NHỊ THỨC (BINOMIAL DISTRIBUTION)
    avg_rating = (r1*1 + r2*2 + r3*3 + r4*4 + r5*5) / total_ratings if total_ratings > 0 else 3.5
    quality_multiplier = avg_rating / 5.0
    
    price_multiplier = 1.0
    if price > 150000: price_multiplier = 0.6     
    elif price < 80000: price_multiplier = 1.3    

    # Tính xác suất (Probability)
    wishlist_prob = min(np.random.uniform(0.02, 0.05) * quality_multiplier, 1.0)
    purchase_prob = min(np.random.uniform(0.005, 0.025) * price_multiplier * quality_multiplier, 1.0)

    # Binomial sẽ "tung đồng xu" đánh giá từng lượt view một. 
    # Nếu 1 sách chỉ có 12 views, tung 12 lần với xác suất mua 1.5% -> Gần như 90% sách này sẽ có 0 Purchase.
    wishlists_7d = np.random.binomial(views_7d, wishlist_prob)
    purchases_7d = np.random.binomial(views_7d, purchase_prob)

    # Cộng dồn số cũ với kết quả tung đồng xu của lượng View chênh lệch (30d - 7d)
    wishlists_30d = wishlists_7d + np.random.binomial(views_30d - views_7d, wishlist_prob)
    purchases_30d = purchases_7d + np.random.binomial(views_30d - views_7d, purchase_prob)

    return pd.Series({
        'views_7d': views_7d,
        'favorite_7d': wishlists_7d,
        'purchases_7d': purchases_7d,
        'views_30d': views_30d,
        'favorite_30d': wishlists_30d,
        'purchases_30d': purchases_30d
    })

# Chạy cập nhật lại
new_columns = books_with_tags.apply(simulate_realistic_tracking, axis=1)

# Xóa cột cũ để join lại cho chuẩn
cols_to_drop = ['views_7d', 'favorite_7d','purchases_7d', 'views_30d', 'favorite_30d','purchases_30d']
existing_cols = [c for c in cols_to_drop if c in books_with_tags.columns]
if existing_cols:
    books_with_tags.drop(columns=existing_cols, inplace=True)

books_with_tags = books_with_tags.join(new_columns)

# ─── KIỂM TRA ĐỘ CHÂN THỰC ───
# Đếm xem có bao nhiêu sách bị "ế" hoàn toàn (0 lượt mua trong 7 ngày)
zero_purchase_books = (books_with_tags['purchases_7d'] == 0).sum()
zero_view_books = (books_with_tags['views_7d'] == 0).sum()

print(f"Số sách 0 view trong tuần: {zero_view_books} / {len(books_with_tags)} cuốn")
print(f"Số sách 0 lượt mua trong tuần: {zero_purchase_books} / {len(books_with_tags)} cuốn")

Số sách 0 view trong tuần: 55 / 10000 cuốn
Số sách 0 lượt mua trong tuần: 3724 / 10000 cuốn


In [16]:
cols_to_drop = ['Views_7D', 'Purchases_7D','Wishlists_7D', 'Views_30D', 'Purchases_30D','Wishlists_30D']
existing_cols = [c for c in cols_to_drop if c in books_with_tags.columns]
if existing_cols:
    books_with_tags.drop(columns=existing_cols, inplace=True)


In [26]:
import numpy as np

# 1. Tính tổng số lượt đánh giá (total_ratings)
# Cộng ngang các cột ratings_1 đến ratings_5 của mỗi dòng (axis=1)
books_with_tags['total_ratings'] = books_with_tags[['ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5']].sum(axis=1)

# 2. Xử lý chia cho 0: Tạo một mảng an toàn để chia, nếu total_ratings = 0 thì tạm coi là 1 để tránh lỗi NaN
safe_total = books_with_tags['total_ratings'].replace(0, 1)

# 3. Tính điểm trung bình (average_rating)
# Điểm trung bình = (1*r1 + 2*r2 + 3*r3 + 4*r4 + 5*r5) / total_ratings
# Dùng hàm round(2) để làm tròn đến 2 chữ số thập phân (VD: 4.25)
books_with_tags['average_rating'] = round(
    (
        books_with_tags['ratings_1'] * 1 + 
        books_with_tags['ratings_2'] * 2 + 
        books_with_tags['ratings_3'] * 3 + 
        books_with_tags['ratings_4'] * 4 + 
        books_with_tags['ratings_5'] * 5
    ) / safe_total, 
    2
)

# Nếu total_ratings = 0, gán average_rating bằng 0 (hoặc None tùy hệ thống của bạn)
books_with_tags.loc[books_with_tags['total_ratings'] == 0, 'average_rating'] = 0.0

# In thử 5 dòng đầu tiên để xem kết quả
books_with_tags[['ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'total_ratings', 'average_rating']].head(10)

,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,total_ratings,average_rating
0,66715,127936,560092,1481305,2706317,4942365,4.34
1,75504,101676,455024,1156318,3011543,4800065,4.44
2,456191,436802,793319,875073,1355439,3916824,3.57
3,60427,117415,446835,1001952,1714267,3340896,4.25
4,86236,197621,606158,936012,947718,2773745,3.89
5,47994,92723,327550,698471,1311871,2478609,4.26
6,46023,76784,288649,665635,1119718,2196809,4.25
7,109383,185520,455042,661516,709176,2120637,3.79
8,77841,145740,458429,716569,680175,2078754,3.85
9,54700,86485,284852,609755,1155673,2191465,4.24


In [27]:
books_with_tags.describe()

,book_id,original_publication_year,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,price,pages,readTime,chapters,views_7d,favorite_7d,purchases_7d,views_30d,favorite_30d,purchases_30d,total_ratings,average_rating
count,1.000000e+04,9979.000000,10000.000000,10000.000000,10000.000000,1.000000e+04,1.000000e+04,10000.000000,10000.0,10000.0,10000.0,10000.000000,10000.00000,10000.000000,10000.000000,10000.00000,10000.000000,1.000000e+04,10000.000000
mean,5.264697e+06,1981.987674,1345.040600,3110.885000,11475.893800,1.996570e+04,2.378981e+04,114976.900000,0.0,0.0,0.0,131.960500,3.68110,1.542400,528.473100,14.73890,6.181300,5.968732e+04,4.002191
std,7.575462e+06,152.576665,6635.626263,9717.123578,28546.449183,5.144736e+04,7.976889e+04,33348.267977,0.0,0.0,0.0,126.035892,4.18824,2.024256,505.437516,15.40774,6.920088,1.678038e+05,0.254427
min,1.000000e+00,-1750.000000,11.000000,30.000000,323.000000,7.500000e+02,7.540000e+02,45000.000000,0.0,0.0,0.0,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,5.510000e+03,2.470000
25%,4.627575e+04,1990.000000,196.000000,656.000000,3112.000000,5.405750e+03,5.334000e+03,91000.000000,0.0,0.0,0.0,56.000000,1.00000,0.000000,226.000000,5.00000,2.000000,1.543875e+04,3.850000
50%,3.949655e+05,2004.000000,391.000000,1163.000000,4894.000000,8.269500e+03,8.836000e+03,116000.000000,0.0,0.0,0.0,111.000000,3.00000,1.000000,444.000000,11.00000,4.000000,2.383250e+04,4.020000
75%,9.382225e+06,2011.000000,885.000000,2353.250000,9287.000000,1.602350e+04,1.730450e+04,139000.000000,0.0,0.0,0.0,177.000000,5.00000,2.000000,706.000000,20.00000,9.000000,4.591500e+04,4.180000
max,3.328864e+07,2017.000000,456191.000000,436802.000000,793319.000000,1.481305e+06,3.011543e+06,180000.000000,0.0,0.0,0.0,2232.000000,79.00000,29.000000,10074.000000,267.00000,117.000000,4.942365e+06,4.820000


In [28]:
books_with_tags.columns

Index(['book_id', 'authors', 'original_publication_year', 'original_title',
       'language_code', 'tags', 'ratings_1', 'ratings_2', 'ratings_3',
       'ratings_4', 'ratings_5', 'image_url', 'small_image_url', 'price',
       'mood', 'badge', 'description', 'longDescription', 'pages', 'readTime',
       'status', 'chapters', 'previewText', 'accentColor', 'views_7d',
       'favorite_7d', 'purchases_7d', 'views_30d', 'favorite_30d',
       'purchases_30d', 'total_ratings', 'average_rating'],
      dtype='object')

In [29]:
books_with_tags.to_csv('E:/CodeFolder/Github_project/BookRecProject/Backend/books_with_tags_enhanced_2.csv', index=False)